# 🖌️ Qwen-Image-Edit 2509 — ComfyUI on Colab (A100)

Run **Qwen-Image-Edit 2509** in **ComfyUI** on a Colab **A100**, served through a public URL (Colab proxy by default, with Cloudflare/ngrok options). Models live in Google Drive so they download **once** and are reused on every session.

Qwen-Image-Edit is an **instruction-driven image editor** — change a background, swap an outfit, restyle a scene, add/remove objects — by describing the edit in plain language. The **2509** build accepts **up to 3 input images** (main image + style / background references). This notebook is wired to the bundled workflow `workflows/qwen_image_edit_2509_subgraph.json`, which uses the **4-step Lightning LoRA** for fast (~4 step) edits.

| Component | File | ComfyUI folder |
|---|---|---|
| Diffusion (edit) model | `qwen_image_edit_2509_fp8_e4m3fn.safetensors` (~20 GB) | `diffusion_models` |
| Text encoder | `qwen_2.5_vl_7b_fp8_scaled.safetensors` (~9 GB) | `text_encoders` |
| VAE | `qwen_image_vae.safetensors` (~250 MB) | `vae` |
| Lightning 4-step LoRA | `Qwen-Image-Edit-2509-Lightning-4steps-V1.0-bf16.safetensors` (~850 MB) | `loras` |

> The bundled workflow uses only **ComfyUI core nodes** (`UNETLoader`, `CLIPLoader`, `VAELoader`, `LoraLoaderModelOnly`, `TextEncodeQwenImageEditPlus`, `CFGNorm`, `ModelSamplingAuraFlow`, `KSampler`, …). `TextEncodeQwenImageEditPlus` / `CFGNorm` need a **recent ComfyUI** — Step 3 pulls latest, so restart ComfyUI after updates.

**Setup:** `Runtime → Change runtime type → A100 GPU`. The edit model is fp8 (~20 GB) and wants **≥16 GB VRAM** free; A100 (40 GB) is comfortable, L4 (24 GB) works, T4 (16 GB) is borderline — add `--lowvram` in Step 7 if you hit OOM.

## Step 1 — Verify GPU

In [ ]:
import subprocess
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip()
print(f'GPU: {gpu}')
if 'A100' in gpu:
    print('✅ A100 (40 GB) — ideal for Qwen-Image-Edit 2509 fp8')
elif 'L4' in gpu:
    print('✅ L4 (24 GB) — works with the fp8 model; add --lowvram in Step 7 if OOM')
elif 'T4' in gpu:
    print('⚠️  T4 (16 GB) is borderline for the fp8 edit model. Prefer A100/L4, or use --lowvram in Step 7.')
else:
    print('⚠️  Recommended: A100. Runtime → Change runtime type → A100 GPU')

## Step 2 — Mount Google Drive

Creates persistent model/output folders under `MyDrive/ComfyUI_Qwen`, so already-downloaded weights are reused across sessions and edits are written straight to Drive.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/ComfyUI_Qwen'
MODELS_DIR = f'{DRIVE_BASE}/models'

for d in [f'{MODELS_DIR}/diffusion_models', f'{MODELS_DIR}/text_encoders',
          f'{MODELS_DIR}/vae', f'{MODELS_DIR}/loras',
          f'{DRIVE_BASE}/output', f'{DRIVE_BASE}/input_images']:
    os.makedirs(d, exist_ok=True)

print(f'✅ Drive mounted: {DRIVE_BASE}')

## Step 3 — Install ComfyUI + custom nodes

Native Qwen-Image-Edit support is built into recent ComfyUI core. Pulls latest ComfyUI (so `TextEncodeQwenImageEditPlus` / `CFGNorm` are present) and adds ComfyUI-Manager.

In [ ]:
import os, subprocess
os.chdir('/content')

# ComfyUI — clone fresh if missing/corrupt, else update
if os.path.exists('/content/ComfyUI'):
    ok = subprocess.run(['git', 'rev-parse', '--git-dir'], cwd='/content/ComfyUI',
                        capture_output=True).returncode == 0
    if ok:
        !cd /content/ComfyUI && git pull -q
        print('✅ ComfyUI updated')
    else:
        !rm -rf /content/ComfyUI && git clone -q https://github.com/comfyanonymous/ComfyUI.git
        print('✅ ComfyUI re-cloned (was corrupt)')
else:
    !git clone -q https://github.com/comfyanonymous/ComfyUI.git
    print('✅ ComfyUI cloned')

# Core requirements (flag avoids reinstalling every session)
if not os.path.exists('/content/comfyui_reqs_installed'):
    !pip install -q -r /content/ComfyUI/requirements.txt
    open('/content/comfyui_reqs_installed', 'w').close()
    print('✅ Requirements installed')
else:
    print('✅ Requirements already installed')

# Custom nodes (ComfyUI-Manager for easy node/model management)
for name, repo in [('ComfyUI-Manager', 'https://github.com/ltdrdata/ComfyUI-Manager.git')]:
    path = f'/content/ComfyUI/custom_nodes/{name}'
    if not os.path.exists(path):
        !git clone -q {repo} {path}
        if os.path.exists(f'{path}/requirements.txt'):
            !pip install -q -r {path}/requirements.txt
        print(f'✅ {name} installed')
    else:
        !cd {path} && git pull -q
        print(f'✅ {name} ready')

print('\n✅ All installs complete')

## Step 4 — Link Drive model folders into ComfyUI

Symlinks each `models/<folder>` plus `output` and `input` to Drive, so ComfyUI reads the already-downloaded weights, writes edits straight to Drive, and keeps uploaded source images persistent.

In [ ]:
import os
COMFY_MODELS = '/content/ComfyUI/models'

links = {f'{COMFY_MODELS}/{f}': f'{MODELS_DIR}/{f}'
         for f in ['diffusion_models', 'text_encoders', 'vae', 'loras']}
links['/content/ComfyUI/output'] = f'{DRIVE_BASE}/output'
links['/content/ComfyUI/input'] = f'{DRIVE_BASE}/input_images'

for dst, src in links.items():
    if os.path.exists(dst) and not os.path.islink(dst):
        !rm -rf {dst}
    if not os.path.islink(dst):
        os.symlink(src, dst)
    print(f'  ✅ {os.path.basename(dst)} → Drive')
print('\n✅ Folders linked')

## Step 5 — Download Qwen-Image-Edit models (skips anything already in Drive)

First-ever run pulls ~30 GB (~20–30 min). After that, everything reports **present** and downloads nothing. All four files are required by the bundled workflow (the Lightning LoRA drives the 4-step KSampler).

In [ ]:
import os
EDIT = 'https://huggingface.co/Comfy-Org/Qwen-Image-Edit_ComfyUI/resolve/main/split_files'
BASE = 'https://huggingface.co/Comfy-Org/Qwen-Image_ComfyUI/resolve/main/split_files'
LX2V = 'https://huggingface.co/lightx2v/Qwen-Image-Lightning/resolve/main'

models = [
    ('Qwen-Image-Edit 2509 fp8 (~20GB)',
     f'{EDIT}/diffusion_models/qwen_image_edit_2509_fp8_e4m3fn.safetensors',
     f'{MODELS_DIR}/diffusion_models/qwen_image_edit_2509_fp8_e4m3fn.safetensors'),
    ('Text Encoder Qwen2.5-VL 7B fp8 (~9GB)',
     f'{BASE}/text_encoders/qwen_2.5_vl_7b_fp8_scaled.safetensors',
     f'{MODELS_DIR}/text_encoders/qwen_2.5_vl_7b_fp8_scaled.safetensors'),
    ('VAE qwen_image_vae (~250MB)',
     f'{BASE}/vae/qwen_image_vae.safetensors',
     f'{MODELS_DIR}/vae/qwen_image_vae.safetensors'),
    ('Lightning 4-step Edit LoRA (~850MB)',
     f'{LX2V}/Qwen-Image-Edit-2509/Qwen-Image-Edit-2509-Lightning-4steps-V1.0-bf16.safetensors',
     f'{MODELS_DIR}/loras/Qwen-Image-Edit-2509-Lightning-4steps-V1.0-bf16.safetensors'),
]

for label, url, dest in models:
    if os.path.exists(dest) and os.path.getsize(dest) > 1024:
        print(f'  ✅ Present ({os.path.getsize(dest)/1024**3:.2f}GB): {label}')
    else:
        print(f'  ⬇️  Downloading: {label}')
        !wget -q --show-progress -O "{dest}" "{url}"
        print(f'  ✅ Done ({os.path.getsize(dest)/1024**3:.2f}GB): {label}')
print('\n✅ All models ready')

## Step 5b — *(Optional)* Add your own LoRAs

Two ways to add LoRAs, both land in Drive `ComfyUI_Qwen/models/loras/` and show up in the `LoraLoaderModelOnly` dropdown after a ComfyUI restart:

1. **Drop-in:** copy any `.safetensors` LoRA into Drive → `ComfyUI_Qwen/models/loras/`. Nothing to run.
2. **Auto-download:** add entries to `EXTRA_LORAS` below (a direct URL + the filename to save). Works with any Hugging Face `resolve/main/...` link. Then in the workflow, add / point a `LoraLoaderModelOnly` at it (chain multiple LoRAs by stacking `LoraLoaderModelOnly` nodes; keep strengths modest, e.g. 0.6–1.0).

In [ ]:
import os

# Add your LoRAs here → ('https://.../file.safetensors', 'saved_name.safetensors')
EXTRA_LORAS = [
    # ('https://huggingface.co/Comfy-Org/Qwen-Image-Edit_ComfyUI/resolve/main/split_files/loras/Qwen-Image-Edit-2509-Relight.safetensors',
    #  'Qwen-Image-Edit-2509-Relight.safetensors'),
]

lora_dir = f'{MODELS_DIR}/loras'
os.makedirs(lora_dir, exist_ok=True)

if not EXTRA_LORAS:
    print('No extra LoRAs listed. Drop .safetensors files into Drive '
          f'{lora_dir}/ to add them, or edit EXTRA_LORAS above.')
for url, name in EXTRA_LORAS:
    dest = f'{lora_dir}/{name}'
    if os.path.exists(dest) and os.path.getsize(dest) > 1024:
        print(f'  ✅ Present ({os.path.getsize(dest)/1024**2:.0f}MB): {name}')
    else:
        print(f'  ⬇️  Downloading: {name}')
        !wget -q --show-progress -O "{dest}" "{url}"
        print(f'  ✅ Done ({os.path.getsize(dest)/1024**2:.0f}MB): {name}')

print('\nInstalled LoRAs in Drive models/loras/:')
for f in sorted(os.listdir(lora_dir)):
    if f.endswith('.safetensors'):
        print(f'  • {f}')
print('\n↻ Restart ComfyUI (re-run Step 7) for new LoRAs to appear in the dropdown.')

## Step 6 — Install the bundled workflow

Copies `workflows/qwen_image_edit_2509_subgraph.json` (this repo) into ComfyUI's user workflows so it appears under **Workflows** (📂 sidebar) in the UI. Clones this repo into Colab if needed.

In [ ]:
import os, shutil

REPO_URL = 'https://github.com/mmorrisj/qwen_edit.git'
REPO_DIR = '/content/qwen_edit'
if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull -q

src = f'{REPO_DIR}/workflows/qwen_image_edit_2509_subgraph.json'
wf_dir = '/content/ComfyUI/user/default/workflows'
os.makedirs(wf_dir, exist_ok=True)
if os.path.exists(src):
    shutil.copy(src, f'{wf_dir}/qwen_image_edit_2509_subgraph.json')
    print('✅ Workflow installed → open it from the Workflows (📂) sidebar in ComfyUI')
else:
    print('⚠️  Workflow file not found in repo; load it manually via Workflow → Open.')

## Step 7 — Launch ComfyUI + public URL

Starts ComfyUI, waits until it's actually serving, then exposes it. Pick a `TUNNEL` method:

- **`colab`** *(default — no auth)*: opens ComfyUI as a **clickable "new window" link + an embedded iframe** inside this cell's output. ⚠️ Do **not** copy the raw `...prod.colab.dev` URL into a separate browser — it only works inside this Colab session and otherwise returns **HTTP 404**. Use the link/iframe this cell renders.
- **`cloudflare`**: quick `trycloudflare.com` tunnel that works in any browser. The cell first deletes any stale `~/.cloudflared` credentials (the usual cause of *"authentication"* errors) and installs a fresh binary.
- **`ngrok`**: paste a free authtoken from [ngrok dashboard](https://dashboard.ngrok.com/get-started/your-authtoken).

If ComfyUI itself fails to start, the tail of its log is shown. **Keep this cell running.**

In [ ]:
#@title Step 7 — Launch ComfyUI + tunnel { display-mode: "form" }
TUNNEL = "colab"  #@param ["colab", "cloudflare", "ngrok"]
NGROK_TOKEN = ""  #@param {type:"string"}

import os, re, time, subprocess, urllib.request

PORT = 8188
LOG = '/tmp/comfyui.log'

# --- start ComfyUI (kill any previous run first) ---
subprocess.run(['pkill', '-f', 'main.py'], capture_output=True)
subprocess.run(['pkill', '-f', 'cloudflared'], capture_output=True)
time.sleep(2)

# --enable-cors-header '*' relaxes cross-origin checks for proxied access.
# Add '--lowvram' to the list below if you hit OOM on a 16-24 GB GPU.
comfy = subprocess.Popen(
    ['python', 'main.py', '--listen', '127.0.0.1', '--port', str(PORT),
     '--preview-method', 'auto', '--enable-cors-header', '*'],
    cwd='/content/ComfyUI', stdout=open(LOG, 'w'), stderr=subprocess.STDOUT)

print('Waiting for ComfyUI to start...')
ready = False
for _ in range(60):
    time.sleep(2)
    if comfy.poll() is not None:
        print('\n❌ ComfyUI exited. Last log lines:\n')
        print(subprocess.run(['tail', '-n', '40', LOG], capture_output=True, text=True).stdout)
        raise SystemExit('ComfyUI failed to start — see log above.')
    try:
        if urllib.request.urlopen(f'http://127.0.0.1:{PORT}/system_stats', timeout=2).status == 200:
            ready = True
            break
    except Exception:
        pass
if not ready:
    print(subprocess.run(['tail', '-n', '40', LOG], capture_output=True, text=True).stdout)
    raise SystemExit('ComfyUI did not become ready in time — see log above.')
print('✅ ComfyUI is serving on :%d' % PORT)

def banner(url, note=''):
    print('\n' + '=' * 64)
    print('  🚀  Open ComfyUI:  ' + url)
    if note:
        print('  ' + note)
    print('=' * 64)

tunnel = None

if TUNNEL == 'colab':
    # Same-origin embed/link — avoids both the 404 (raw proxy URL pasted in a new
    # browser) and ComfyUI's 403 host/origin check. RECOMMENDED on Colab.
    from google.colab import output
    print('▶ Click this link to open ComfyUI in a new tab:')
    output.serve_kernel_port_as_window(PORT)
    print('\n▶ ...or use ComfyUI embedded right here:')
    output.serve_kernel_port_as_iframe(PORT, height='820')

elif TUNNEL == 'ngrok':
    # ngrok forwards Host == its own domain == Origin, so ComfyUI's host/origin
    # check passes. Reliable public URL that works in any browser.
    if not NGROK_TOKEN:
        raise SystemExit('Set NGROK_TOKEN in the form (free at dashboard.ngrok.com), or use TUNNEL="colab".')
    !pip install -q pyngrok
    from pyngrok import ngrok, conf
    conf.get_default().auth_token = NGROK_TOKEN
    ngrok.kill()
    banner(ngrok.connect(PORT, 'http').public_url, '(ngrok)')

elif TUNNEL == 'cloudflare':
    # Quick tunnel. ComfyUI v1.19+ may 403 ("non matching host and origin")
    # through a proxy; --http-host-header keeps the forwarded Host aligned, and a
    # stale cookie is the other common cause — open the link in an incognito tab.
    # If it still 403s, switch TUNNEL to "ngrok" or "colab".
    !rm -rf ~/.cloudflared
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared
    tunnel = subprocess.Popen(
        ['cloudflared', 'tunnel', '--no-autoupdate',
         '--url', f'http://127.0.0.1:{PORT}', '--http-host-header', f'127.0.0.1:{PORT}'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    url_re = re.compile(r'https://[a-z0-9\-]+\.trycloudflare\.com')
    found = False
    t0 = time.time()
    for line in tunnel.stdout:
        m = url_re.search(line)
        if m:
            banner(m.group(0), '403? open in an incognito tab; still 403 → use TUNNEL="ngrok" or "colab"')
            found = True
            break
        if time.time() - t0 > 40:
            break
    if not found:
        print('⚠️  Cloudflare returned no URL. Set TUNNEL="colab" or "ngrok" and re-run.')

print('\n⏳ Keep this cell running. Interrupt to stop.')
try:
    comfy.wait()
except KeyboardInterrupt:
    comfy.terminate()
    if tunnel:
        tunnel.terminate()
    print('\n🛑 ComfyUI stopped')

## Step 8 — Run the edit workflow

In the ComfyUI tab:
1. **Open the workflow:** Workflows (📂 sidebar) → `qwen_image_edit_2509_subgraph`. *(Or `Workflow → Browse Templates → Image → Qwen-Image-Edit`.)*
2. Confirm the loaders inside the **Qwen Image Edit 2509** subgraph point at your files: `UNETLoader` = `qwen_image_edit_2509_fp8_e4m3fn.safetensors`, `CLIPLoader` = `qwen_2.5_vl_7b_fp8_scaled` (type `qwen_image`), `VAELoader` = `qwen_image_vae.safetensors`, `LoraLoaderModelOnly` = `Qwen-Image-Edit-2509-Lightning-4steps-V1.0-bf16.safetensors`.
3. **Load Image** node(s) = your source image(s). Upload via the UI, or drop files in Drive `ComfyUI_Qwen/input_images/` (see utility below). 2509 accepts **up to 3** — main image + optional references.
4. Write the edit in the prompt, e.g. *"Remove the yellow balloon"*, *"Change the balloon's color to blue"*, *"Replace the man with a child, keep the same oil-painting style"*.
5. **Settings (with the 4-step Lightning LoRA, as bundled):** steps **4**, CFG **1.0**, sampler **euler**, scheduler **simple**. *For max quality without the LoRA:* bypass the `LoraLoaderModelOnly`, then steps **20**, CFG **2.5** (fp8) or steps **50**, CFG **4.0** (bf16).
6. **Run.** The edited image lands in Drive `ComfyUI_Qwen/output/`.

**OOM?** Add `--lowvram` to the launch command in Step 7, or shrink the input via the `ImageScaleToTotalPixels` node already in the graph.

---
## 🔧 Utilities

### Copy an input image into ComfyUI

In [ ]:
import shutil, os
SOURCE = f'{DRIVE_BASE}/input_images/my_image.png'  # update filename
if os.path.exists(SOURCE):
    # /content/ComfyUI/input is symlinked to Drive input_images, so it's already
    # visible in the Load Image node. This just confirms the file is present.
    print(f'✅ Available in Load Image node: {os.path.basename(SOURCE)}')
else:
    print(f'⚠️  Not found: {SOURCE}\n   Upload to Drive → ComfyUI_Qwen → input_images (or use the UI upload button)')

### List recent edits (outputs)

In [ ]:
import glob, os
outs = sorted(glob.glob(f'{DRIVE_BASE}/output/*.png'), key=os.path.getmtime, reverse=True)
print(f'Found {len(outs)} output image(s):')
for p in outs[:10]:
    print(f'  {os.path.getsize(p)/1024:.0f} KB  {os.path.basename(p)}')
if outs:
    try:
        from IPython.display import Image, display
        print('\nMost recent:')
        display(Image(outs[0]))
    except Exception:
        pass

---
## 📋 Prompt & troubleshooting tips

**Editing prompts** — describe the change, not the whole scene. Qwen-Image-Edit follows edit instructions well:
- Object: `"remove the yellow balloon"`, `"add a small dog next to the man"`
- Recolor / material: `"change the balloon's color to reflective blue"`
- Restyle: `"convert to a realistic photo, keep composition"`, `"make it an oil painting"`
- Background: `"replace the background with a sunset beach"`
- Text: Qwen is strong at rendering/editing text — `"change the sign to read 'OPEN'"`
- Multi-image (2509): image 1 = subject, image 2 = style/background reference, image 3 = extra reference

**Troubleshooting**
- **`TextEncodeQwenImageEditPlus` / `CFGNorm` missing** → ComfyUI is out of date. Re-run Step 3 (`git pull`), then restart ComfyUI (re-run Step 7).
- **Cloudflare "authentication" / 403** → use `TUNNEL = "colab"` in Step 7 (no auth), or open the Cloudflare link in an incognito tab. The Cloudflare path also wipes stale `~/.cloudflared` creds.
- **OOM (out of memory)** → add `--lowvram` to the launch command in Step 7; keep input images ≤ ~1–2 MP (the `ImageScaleToTotalPixels` node caps this).
- **Wrong CLIP type** → `CLIPLoader` type must be `qwen_image`.
- **Cell stops immediately** → Step 7 prints the ComfyUI log tail on failure; read it for the real error.